Task: Restaurant Recommendation

Objective: Create a restaurant recommendation
system based on user preferences.

Steps:
Preprocess the dataset by handling missing
values and encoding categorical variables.

Determine the criteria for restaurant
recommendations (e.g., cuisine preference,
price range).

Implement a content-based filtering
approach where users are recommended
restaurants similar to their preferred criteria.

Test the recommendation system by
providing sample user preferences and
evaluating the quality of recommendations.

In [32]:
#import requierd modules
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer as tfidf # it is used to convert a collection of raw documents to a matrix of TF-IDF features   
from sklearn.metrics.pairwise import cosine_similarity # it is used to calculate the similarity between two vectors



In [33]:
#load data
df=pd.read_csv('Dataset.csv')
df.head(1)

,Restaurant ID,Restaurant Name,Country Code,City,Address,Locality,Locality Verbose,Longitude,Latitude,Cuisines,Average Cost for two,Currency,Has Table booking,Has Online delivery,Is delivering now,Switch to order menu,Price range,Aggregate rating,Rating color,Rating text,Votes
0,6317637,Le Petit Souffle,162,Makati City,"Third Floor, Century City Mall, Kalayaan Avenu...","Century City Mall, Poblacion, Makati City","Century City Mall, Poblacion, Makati City, Mak...",121.027535,14.565443,"French, Japanese, Desserts",1100,Botswana Pula(P),Yes,No,No,No,3,4.8,Dark Green,Excellent,314


In [34]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9551 entries, 0 to 9550
Data columns (total 21 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Restaurant ID         9551 non-null   int64  
 1   Restaurant Name       9551 non-null   str    
 2   Country Code          9551 non-null   int64  
 3   City                  9551 non-null   str    
 4   Address               9551 non-null   str    
 5   Locality              9551 non-null   str    
 6   Locality Verbose      9551 non-null   str    
 7   Longitude             9551 non-null   float64
 8   Latitude              9551 non-null   float64
 9   Cuisines              9542 non-null   str    
 10  Average Cost for two  9551 non-null   int64  
 11  Currency              9551 non-null   str    
 12  Has Table booking     9551 non-null   str    
 13  Has Online delivery   9551 non-null   str    
 14  Is delivering now     9551 non-null   str    
 15  Switch to order menu  9551 non-n

In [35]:
df.isnull().sum()

Restaurant ID           0
Restaurant Name         0
Country Code            0
City                    0
Address                 0
Locality                0
Locality Verbose        0
Longitude               0
Latitude                0
Cuisines                9
Average Cost for two    0
Currency                0
Has Table booking       0
Has Online delivery     0
Is delivering now       0
Switch to order menu    0
Price range             0
Aggregate rating        0
Rating color            0
Rating text             0
Votes                   0
dtype: int64

In [36]:
#select only required columns
df_rec=df[['Restaurant ID','Restaurant Name','Cuisines','City','Votes','Price range','Aggregate rating']]
#fill na values of Cuisines column with 'unknown' beacuse it is a categorical column and we can not fill it with mean or median
df_rec['Cuisines']=df_rec['Cuisines'].fillna('unknown',inplace=True)
#replace commas with space in Cuisines column because we want to treat each cuisine as a separate word
df_rec['clean_Cuisines']=df_rec['Cuisines'].str.replace(',',' ')
#combine relevant textual features into a single column for vectorization
df_rec['combined_features'] = df_rec['Restaurant Name'] + ' ' + df_rec['clean_Cuisines'] + ' ' + df_rec['City']+ ' ' + df_rec['Price range'].astype(str)
df_rec.head(1)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_616\3178384040.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_rec['Cuisines']=df_rec['Cuisines'].fillna('unknown',inplace=True)


,Restaurant ID,Restaurant Name,Cuisines,City,Votes,Price range,Aggregate rating,clean_Cuisines,combined_features
0,6317637,Le Petit Souffle,"French, Japanese, Desserts",Makati City,314,3,4.8,French Japanese Desserts,Le Petit Souffle French Japanese Desserts Ma...


In [37]:
df_rec['clean_Cuisines'] 

0             French  Japanese  Desserts
1                               Japanese
2       Seafood  Asian  Filipino  Indian
3                        Japanese  Sushi
4                       Japanese  Korean
                      ...               
9546                             Turkish
9547     World Cuisine  Patisserie  Cafe
9548              Italian  World Cuisine
9549                     Restaurant Cafe
9550                                Cafe
Name: clean_Cuisines, Length: 9551, dtype: str

In [38]:
#Defining the recommendation function
tfidf = tfidf(stop_words='english') #initialize the vectorizer with English stop words
tfidf_matrix = tfidf.fit_transform(df_rec['combined_features']) #fit and transform the combined_features column to create a TF-IDF matrix
def recommend_restaurants(user_city,user_cuisine,user_maxprice,min_rating=3.5,top=5):
    #format the user input to match the format of the combined_features column
    user_input = f"{user_city} {user_cuisine} {user_maxprice}"
    user_vec = tfidf.transform([user_input]) #transform the user input into a vector using the same vectorizer
    #calculate the cosine similarity between the user input and all restaurants.
    similarities = cosine_similarity(user_vec, tfidf_matrix).flatten()
    #assign computed similarity scores back to a working data frame
    df_result=df_rec.copy()
    df_result['similarity'] = similarities[0]
    # apply threshold filters for rating and price range
    filtered_df = df_result[(df_result['Aggregate rating'] >= min_rating) & (df_result['Price range'] <= user_maxprice)]
    #sort the filtered data frame by similarity score in descending order and select the top N results
    sorted_df = filtered_df.sort_values(by=['similarity','Votes','Aggregate rating'], ascending=[False,False,False]).head(top)
    #extract the relevant columns to display as recommendations
    recommendations = sorted_df[['Restaurant Name', 'Cuisines', 'City', 'Price range', 'Aggregate rating','Votes']].head(top)
    return recommendations

In [41]:
#test and evalution of the recommendation function
#define sample user inputs
user_city = input("Enter the city you are in: ")
user_cuisine = input("Enter the cuisine you like: ")
user_maxprice = int(input("Enter the maximum price range you can afford: "))
min_rating = float(input("Enter the minimum rating you prefer: "))
recommendations = recommend_restaurants(user_city, user_cuisine, user_maxprice, min_rating)

# display the recommendations
print("Top 5 Restaurant Recommendations:")
print(recommendations.to_string(index=False))

Top 5 Restaurant Recommendations:
          Restaurant Name                                   Cuisines      City  Price range  Aggregate rating  Votes
                     Toit                   Italian, American, Pizza Bangalore            4               4.8  10934
                 Truffles                     American, Burger, Cafe Bangalore            2               4.7   9667
         Hauz Khas Social Continental, American, Asian, North Indian New Delhi            3               4.3   7931
                Peter Cat                  Continental, North Indian   Kolkata            3               4.3   7574
AB's - Absolute Barbecues      European, Mediterranean, North Indian Bangalore            3               4.6   6907
